# HW2 실험 실행 (VS Code + Colab 커널)

오른쪽 위 **Select Kernel → Colab → New Colab Server → GPU → T4**를 고른 뒤 위에서부터 실행합니다.
코드는 2번 셀에 들어 있어서 파일을 따로 올릴 필요가 없습니다. 실행 결과는 이 노트북에 저장되므로 Claude가 바로 읽을 수 있습니다.

In [ ]:
import torch, platform
assert torch.cuda.is_available(), 'GPU가 없습니다. Colab 서버를 T4 GPU로 다시 만드세요.'
print(torch.cuda.get_device_name(0), '|', platform.node())

In [ ]:
# 실험 코드 풀기 (hw2_code 폴더의 압축본. 코드가 바뀌면 Claude가 이 셀을 다시 만듭니다)
import base64, io, os, zipfile
BLOB = '''
...
AOcPAgAAAA==
'''
zipfile.ZipFile(io.BytesIO(base64.b64decode(BLOB.replace('\n', '')))).extractall('/content')
os.chdir('/content/hw2_code')
OUT = '/content/hw2_results'
os.makedirs(OUT, exist_ok=True)
print(sorted(os.listdir('.')))

In [ ]:
!pip -q install -r requirements.txt
!python -c "import torch, transformers, datasets; print(torch.__version__, transformers.__version__, datasets.__version__)"

## 1. 파일럿 (학습 분할 50문항씩, 1,320회, 약 1–2분)

평정으로 선정한 문구(Q02, Q04, Q03, Q01) 중 앞의 두 묶음으로 돌립니다. 끝나면 아래 분석 셀에서 통과 여부(`neutral_pass`, `extraction_pass`)가 모두 `true`인지 확인합니다.

In [ ]:
# 이전 파일럿 결과를 지우고 새 코드로 다시 실행
!rm -f /content/hw2_results/*pilot*
!cd /content/hw2_code && python -m kspeech.run --phase pilot --out /content/hw2_results 2>&1 | grep -v Warning | tail -60

In [ ]:
!cd /content/hw2_code && python -m kspeech.analyze --phase pilot --out /content/hw2_results

## 2. 본실험 (검증 분할, 약 28,000회, 10–20분 예상)

파일럿 통과 여부가 모두 `true`일 때만 실행합니다. 중간에 끊기면 같은 셀을 다시 실행하면 이어서 진행합니다.

In [ ]:
!cd /content/hw2_code && python -m kspeech.run --phase main --out /content/hw2_results 2>&1 | grep --line-buffered -v Warning

In [ ]:
!cd /content/hw2_code && python -m kspeech.analyze --phase main --out /content/hw2_results
!cd /content/hw2_code && python -m kspeech.extra --out /content/hw2_results

## 3. 결과 내보내기

마지막 셀까지 실행한 뒤 노트북을 저장(Cmd+S)합니다.

In [ ]:
# 결과 내보내기: 결과 폴더를 압축해 이 셀의 출력에 글자로 남깁니다 (Claude가 노트북에서 복원)
import base64, io, tarfile
b = io.BytesIO()
with tarfile.open(fileobj=b, mode='w:gz') as t:
    t.add('/content/hw2_results', arcname='hw2_results')
s = base64.b64encode(b.getvalue()).decode()
print('RESULTS_B64_BEGIN'); print(s); print('RESULTS_B64_END'); print(len(s), 'chars')